# Programação Orientada a Objetos com Python — Parte 3
## Herança, Polimorfismo e Composição

**Tema: O Podrão — Sistema de Lanchonete / Food Truck**

**Disciplina:** COMP0496 — Programação A
**Duração:** 200 minutos (Aula 3 de 4 do módulo de POO)

---

### Pré-requisitos

Você deve ter concluído os notebooks **Parte 1** e **Parte 2** e estar confortável com:

- Definir classes, `__init__`, `self`, atributos de instância e de classe.
- Usar `@property` e setters com validação.
- Implementar os principais métodos especiais (`__str__`, `__repr__`, `__eq__`, `__lt__`, `__len__`).

### Objetivos desta aula

Ao final desta aula, você será capaz de:

1. Reconhecer quando uma hierarquia de classes é apropriada (relação **"é um"**).
2. Usar **herança simples** com `super()` para reaproveitar código sem duplicação.
3. **Sobrescrever métodos** em subclasses e estender (não só substituir) o comportamento do pai.
4. Aplicar **polimorfismo** e *duck typing* para escrever código genérico.
5. Distinguir quando usar **herança** vs **composição** (relação "é um" vs "tem um").
6. Entender o básico de **herança múltipla** e **MRO** (*Method Resolution Order*).
7. Definir **classes abstratas** com `abc.ABC` e `@abstractmethod` para estabelecer contratos.

### Como usar este material

Siga as células na ordem. Faça os exercícios antes de avançar. Os blocos ⚠️ marcam pegadinhas importantes.

---

## 1. Revisão rápida da Aula 2

Até agora você sabe construir classes com atributos protegidos, propriedades e dunders. O problema que vamos atacar hoje é **outro**: código **duplicado** entre classes parecidas.

Considere três tipos de lanche do Podrão: `XTudo`, `Dogao` e `Acai`. Todos têm nome, preço e ingredientes. Todos precisam ser descritos e preparados. Se cada um for uma classe completamente independente, vamos acabar reescrevendo a mesma coisa três vezes.

Execute a célula abaixo para sentir o problema:

In [ ]:
# Abordagem ingênua: três classes independentes com muito código duplicado

class XTudo:
    def __init__(self, preco):
        self.nome = "X-Tudo"
        self.preco = preco
        self.ingredientes = "pão, hambúrguer, ovo, queijo, bacon"

    def descrever(self):
        print(f"  🍔 {self.nome} — R${self.preco:.2f}")
        print(f"     Ingredientes: {self.ingredientes}")


class Dogao:
    def __init__(self, preco):
        self.nome = "Dogão"
        self.preco = preco
        self.ingredientes = "pão, salsicha, vinagrete, batata palha"

    def descrever(self):
        print(f"  🍔 {self.nome} — R${self.preco:.2f}")
        print(f"     Ingredientes: {self.ingredientes}")


class Acai:
    def __init__(self, preco):
        self.nome = "Açaí 500ml"
        self.preco = preco
        self.ingredientes = "açaí, granola, banana"

    def descrever(self):
        print(f"  🍔 {self.nome} — R${self.preco:.2f}")
        print(f"     Ingredientes: {self.ingredientes}")


for item in [XTudo(18.50), Dogao(12.00), Acai(15.00)]:
    item.descrever()


O código funciona, mas **cheira mal**:

- O método `descrever()` está duplicado nas três classes. Se quisermos mudar o formato, temos que lembrar de alterar nos três lugares.
- Os atributos `nome`, `preco`, `ingredientes` estão replicados.
- Adicionar um novo tipo de lanche exige recopiar tudo de novo.

A solução clássica para isso é **herança**: colocar o comportamento comum em uma classe "pai" e fazer as classes específicas **herdarem** dela.

---

## 2. Herança simples

**Herança** é o mecanismo pelo qual uma classe *filha* (ou *subclasse*) recebe automaticamente os atributos e métodos de uma classe *pai* (ou *superclasse*), podendo **estender** ou **redefinir** esses comportamentos.

### Quando faz sentido usar herança?

Use quando existe uma relação **"é um"** (do inglês *is-a*) entre os conceitos:

| Relação | Herança? |
|---|---|
| Um `XTudo` **é um** `Lanche` | ✔ Sim |
| Um `Gerente` **é um** `Funcionario` | ✔ Sim |
| Um `Círculo` **é uma** `Forma` | ✔ Sim |
| Um `Pedido` **é uma** `CaixaRegistradora` | ✘ Não (use composição) |
| Um `Lanche` **tem** `Ingredientes` | ✘ Não (use composição) |

```
           ┌───────────────┐
           │    Lanche     │  ← Classe Pai (superclasse)
           │  nome         │
           │  preco        │
           │  ingredientes │
           │  descrever()  │
           └───────┬───────┘
                   │  "é um"
       ┌───────────┼───────────┐
       ▼           ▼           ▼
   ┌─────────┐ ┌─────────┐ ┌─────────┐
   │  XTudo  │ │  Dogao  │ │  Acai   │  ← Subclasses
   └─────────┘ └─────────┘ └─────────┘
```

### 2.1 Sintaxe e `super()`

In [ ]:
# ─── Classe Pai ──────────────────────────────────────────────────────────
class Lanche:
    """Representa um lanche genérico do food truck."""

    def __init__(self, nome, preco, ingredientes):
        self.nome = nome
        self.preco = preco
        self.ingredientes = ingredientes

    def descrever(self):
        print(f"  🍔 {self.nome} — R${self.preco:.2f}")
        print(f"     Ingredientes: {self.ingredientes}")

    def preparar(self):
        print(f"  [COZINHA] Preparando {self.nome}...")


# ─── Classes Filho ───────────────────────────────────────────────────────
class XTudo(Lanche):                         # ← (Lanche) significa "herda de Lanche"
    def __init__(self, extras="bacon extra"):
        # super().__init__(...) chama o construtor do pai.
        # Delegamos a ele a responsabilidade de preencher nome/preco/ingredientes.
        super().__init__(
            nome="X-Tudo",
            preco=18.50,
            ingredientes="pão, hambúrguer, ovo, queijo, bacon, alface, tomate"
        )
        self.extras = extras                 # atributo adicional SÓ do XTudo


class Dogao(Lanche):
    def __init__(self, molhos="ketchup, mostarda, maionese"):
        super().__init__(
            nome="Dogão",
            preco=12.00,
            ingredientes="pão, 2 salsichas, vinagrete, batata palha"
        )
        self.molhos = molhos


class Acai(Lanche):
    def __init__(self, tamanho="500ml"):
        super().__init__(
            nome=f"Açaí {tamanho}",
            preco=15.00 if tamanho == "500ml" else 10.00,
            ingredientes="açaí, granola, banana, leite condensado"
        )
        self.tamanho = tamanho


# ─── Demonstração ────────────────────────────────────────────────────────
x = XTudo("bacon extra, cheddar")
d = Dogao()
a = Acai("700ml")

# Os três herdam `descrever()` e `preparar()` de Lanche — sem duplicação
for item in [x, d, a]:
    item.descrever()
    item.preparar()
    print()

# Também herdam a relação: são instâncias de Lanche
print(f"x é Lanche? {isinstance(x, Lanche)}")
print(f"x é XTudo?  {isinstance(x, XTudo)}")
print(f"d é XTudo?  {isinstance(d, XTudo)}")


**Pontos-chave:**

- `class XTudo(Lanche):` declara que `XTudo` herda de `Lanche`.
- `super().__init__(...)` chama o construtor do pai. Sem isso, os atributos `nome`, `preco` e `ingredientes` não seriam inicializados.
- Os métodos `descrever()` e `preparar()` **não foram redefinidos** em `XTudo` — eles são **herdados automaticamente** do pai.
- `isinstance(x, Lanche)` é `True` porque `XTudo` "é um" `Lanche`.

### 2.2 Sobrescrita (*override*) de métodos

Às vezes queremos que a classe filha **redefina** o comportamento de um método herdado. Isso se chama **sobrescrita** (ou *override*). E, frequentemente, queremos *estender* o comportamento do pai em vez de substituí-lo totalmente — aí usamos `super().metodo()` dentro da sobrescrita.

In [ ]:
class Lanche:
    def __init__(self, nome, preco, ingredientes):
        self.nome = nome
        self.preco = preco
        self.ingredientes = ingredientes

    def descrever(self):
        print(f"  🍔 {self.nome} — R${self.preco:.2f}")
        print(f"     Ingredientes: {self.ingredientes}")


class XTudo(Lanche):
    def __init__(self, extras="bacon extra"):
        super().__init__(
            nome="X-Tudo",
            preco=18.50,
            ingredientes="pão, hambúrguer, ovo, queijo, bacon, alface, tomate"
        )
        self.extras = extras

    def descrever(self):                    # ← SOBRESCRITA
        super().descrever()                 # ← primeiro chama a versão do pai
        print(f"     ⭐ Extras: {self.extras}")   # depois acrescenta o extra


class Acai(Lanche):
    def __init__(self, tamanho="500ml"):
        super().__init__(
            nome=f"Açaí {tamanho}",
            preco=15.00,
            ingredientes="açaí, granola, banana, leite condensado"
        )
        self.tamanho = tamanho

    def descrever(self):
        super().descrever()
        print(f"     📏 Tamanho: {self.tamanho}")


x = XTudo("bacon extra, cheddar")
a = Acai("700ml")

x.descrever()
print()
a.descrever()


O padrão **"estender, não substituir"** (chamar `super().metodo()` antes de acrescentar o comportamento extra) é extremamente comum e mantém o código DRY (*Don't Repeat Yourself*).

### ⚠️ Armadilha: esquecer o `super().__init__()`

Se você sobrescrever `__init__` na subclasse **sem** chamar `super().__init__(...)`, os atributos do pai **não serão inicializados**, e você vai cair em `AttributeError` na primeira vez que tentar usá-los. Veja:

In [ ]:
class LancheBugado(Lanche):
    def __init__(self, extras):
        # ESQUECI de chamar super().__init__()!
        self.extras = extras


bug = LancheBugado("bacon")
try:
    bug.descrever()       # vai falhar: nome, preco, ingredientes não existem
except AttributeError as e:
    print(f"✘ {type(e).__name__}: {e}")
    print("✔ Lição: sempre chame super().__init__() no primeiro parâmetro do filho.")


### Exercício 2.1
Crie uma classe `Bebida` (classe pai) com atributos `nome`, `preco`, `volume_ml` e um método `servir()` que imprima algo como `"Servindo Cajuína (350ml) — R$5.00"`.

Em seguida, crie duas subclasses:

- `Refrigerante(Bebida)` com um atributo adicional `com_gas` (bool).
- `Suco(Bebida)` com um atributo adicional `natural` (bool).

Sobrescreva `servir()` em cada subclasse para incluir a informação extra (ex.: `"...(com gás)"` ou `"...(natural)"`). Use `super().servir()` para não duplicar código.

In [ ]:
# Escreva sua solução aqui



---
## 3. Polimorfismo e *Duck Typing*

**Polimorfismo** (do grego: *muitas formas*) é a capacidade de tratar objetos de tipos diferentes de maneira uniforme. A mesma chamada de método pode desencadear comportamentos diferentes dependendo do tipo real do objeto.

### O princípio fundamental

O código que usa os objetos **não precisa saber** o tipo exato com o qual está lidando — só precisa saber que o objeto *responde à mensagem esperada*.

No nosso food truck, cada tipo de lanche é preparado de um jeito diferente:

- **X-Tudo** → vai pra **chapa** 🔥
- **Dogão** → salsicha vai pra **panela** 🍳
- **Açaí** → vai pro **liquidificador** 🧊

Com polimorfismo, o mesmo laço `for` dispara o comportamento certo para cada um:

In [ ]:
class Lanche:
    def __init__(self, nome, preco):
        self.nome = nome
        self.preco = preco

    def preparar(self):
        # Método "genérico" que será sobrescrito nas subclasses
        print(f"  [{self.nome}] Preparando de forma genérica...")


class XTudo(Lanche):
    def __init__(self):
        super().__init__("X-Tudo", 18.50)

    def preparar(self):
        print(f"  🔥 [CHAPA]         {self.nome}: jogando tudo na chapa quente!")


class Dogao(Lanche):
    def __init__(self):
        super().__init__("Dogão", 12.00)

    def preparar(self):
        print(f"  🍳 [PANELA]        {self.nome}: fervendo a salsicha!")


class Acai(Lanche):
    def __init__(self):
        super().__init__("Açaí 500ml", 15.00)

    def preparar(self):
        print(f"  🧊 [LIQUIDIFICADOR] {self.nome}: batendo com granola e banana!")


# ─── Polimorfismo em ação ────────────────────────────────────────────────
pedido_itens = [XTudo(), Dogao(), Acai(), XTudo()]

print("=== 👨‍🍳 Abrindo a Cozinha ===")
for item in pedido_itens:
    item.preparar()          # mesma chamada → comportamentos diferentes!
print("=== ✅ Tudo pronto! ===")


O laço `for` **não sabe** (e não precisa saber) se está lidando com um `XTudo`, um `Dogao` ou um `Acai`. Ele simplesmente envia a mensagem `preparar()` e cada objeto responde do seu jeito.

**Poder real dessa abordagem:** para adicionar um novo tipo de lanche, basta criar a nova classe com seu próprio `preparar()`. O código do laço **não precisa ser alterado em uma única linha**. Isso é o que torna sistemas orientados a objetos fáceis de estender.

### 3.1 *Duck Typing*: "se parece com pato..."

Python vai um passo além da herança tradicional. A famosa filosofia do *duck typing* diz:

> *"If it walks like a duck and quacks like a duck, it's a duck."*

Ou seja: **Python não verifica o tipo** do objeto antes de chamar um método — ele simplesmente tenta chamar. Se o método existir, funciona; se não existir, dá erro. Isso significa que você pode misturar objetos **sem herança comum** no mesmo laço, desde que todos respondam à mesma "mensagem".

In [ ]:
# Bebida NÃO herda de Lanche — mas também tem um método preparar()
class Bebida:
    def __init__(self, nome):
        self.nome = nome

    def preparar(self):
        print(f"  🥤 [GELADEIRA]     {self.nome}: pegando gelada da geladeira!")


# Misturamos tudo na fila da cozinha — sem herança comum, apenas a "promessa"
# de que todos respondem a preparar()
fila = [XTudo(), Bebida("Cajuína"), Dogao(), Bebida("Guaraná Jesus"), Acai()]

print("=== Processando fila mista ===")
for item in fila:
    item.preparar()


Funciona perfeitamente — mesmo `Bebida` não sendo subclasse de `Lanche`. Em linguagens como Java ou C++, isso exigiria uma *interface* comum declarada explicitamente. Em Python, a "interface" é implícita: **"quem tiver `preparar()`, entra na fila"**.

Isso é poderoso, mas tem um preço: se alguém passar um objeto que não tem `preparar()`, o erro só aparece **em tempo de execução**. É por isso que Python também oferece **classes abstratas** — veremos em breve — para declarar contratos explícitos quando isso for desejável.

### Exercício 3.1
Estenda a classe `Bebida` do exercício anterior adicionando um método `preparar()`. Crie uma lista misturando instâncias de `Lanche` (subclasses) e `Bebida` (subclasses). Chame `preparar()` em cada uma dentro de um único laço `for`. Observe que o laço não precisa fazer nenhum tipo de verificação.

In [ ]:
# Escreva sua solução aqui



---
## 4. Composição: a alternativa à herança

Nem toda relação entre classes é do tipo "é um". Muitas vezes, a relação correta é **"tem um"** (do inglês *has-a*) — e aí você **não** deve usar herança. Você deve usar **composição**: uma classe contém objetos de outras classes como atributos.

### Relação "é um" vs "tem um"

| Relação | Mecanismo |
|---|---|
| Um `XTudo` **é um** `Lanche` | Herança ✔ |
| Um `Pedido` **tem** `Lanches` (uma lista) | Composição ✔ |
| Um `Pedido` **tem uma** `FormaPagamento` | Composição ✔ |
| Um `Carro` **tem um** `Motor` | Composição ✔ |
| Um `Carro` **é um** `Veículo` | Herança ✔ |

```
┌───────────────────────────────────────────────┐
│                  Pedido                       │
│                                               │
│   self.itens     = [Lanche, Lanche, ...] ───► ┌──────────┐
│                                               │  Lanche  │
│   self.pagamento = FormaPagamento()     ───► │  nome    │
│                                               │  preco   │
│                                          ───► ┌──────────────────┐
│                                               │ FormaPagamento   │
│                                               │ tipo             │
│                                               │ processar()      │
└───────────────────────────────────────────────┘
```

### 4.1 Exemplo: classe `Pedido`

In [ ]:
# ─── Componente independente: Forma de Pagamento ─────────────────────────
class FormaPagamento:
    """Representa uma forma de pagamento aceita pelo food truck."""

    def __init__(self, tipo):
        self.tipo = tipo    # "Pix", "Cartão", "Dinheiro", "Fiado"

    def processar(self, valor):
        if self.tipo == "Fiado":
            print(f"  📝 Anotado no caderninho: R${valor:.2f} (FIADO)")
        elif self.tipo == "Pix":
            print(f"  📱 Pix recebido: R${valor:.2f} — valeu!")
        else:
            print(f"  💳 Pagamento via {self.tipo}: R${valor:.2f} aprovado.")


# ─── Classe composta: Pedido ─────────────────────────────────────────────
class Pedido:
    """Pedido construído por COMPOSIÇÃO de componentes."""

    def __init__(self, cliente, pagamento):
        self.cliente = cliente
        self.itens = []                  # ← composição: lista de Lanches
        self.pagamento = pagamento       # ← composição: FormaPagamento

    def adicionar(self, item):
        self.itens.append(item)
        print(f"  ✔ '{item.nome}' adicionado ao pedido de {self.cliente}.")

    def total(self):
        return sum(item.preco for item in self.itens)

    def preparar_tudo(self):
        """POLIMORFISMO: chama preparar() em cada item sem saber o tipo."""
        print(f"\n--- Cozinha do pedido de {self.cliente} ---")
        for item in self.itens:
            item.preparar()

    def resumo(self):
        print(f"\n{'='*42}")
        print(f"  📋 Pedido de: {self.cliente}")
        print(f"{'='*42}")
        for item in self.itens:
            print(f"  • {item.nome:.<28} R${item.preco:.2f}")
        print(f"{'-'*42}")
        print(f"  TOTAL: R${self.total():.2f}")
        self.pagamento.processar(self.total())
        print(f"{'='*42}")


# ─── Uso ─────────────────────────────────────────────────────────────────
pedido1 = Pedido("João", FormaPagamento("Pix"))
pedido1.adicionar(XTudo())
pedido1.adicionar(Dogao())
pedido1.adicionar(Acai())

pedido1.preparar_tudo()
pedido1.resumo()


Repare nas vantagens:

1. **Componentes independentes**: `FormaPagamento` funciona sozinha e poderia ser usada em qualquer outro sistema (padaria, bar, e-commerce) sem alteração.
2. **Baixo acoplamento**: `Pedido` não precisa saber *como* a forma de pagamento processa — só chama `processar()`.
3. **Flexibilidade**: posso trocar a forma de pagamento de um pedido em tempo de execução. Com herança, isso seria impossível.

### 4.2 Quando preferir composição a herança?

Existe um princípio famoso em design de software:

> **"Favor composition over inheritance."**
> *— Gang of Four, Design Patterns (1994)*

A razão é que **herança cria acoplamento forte**: mudanças na classe pai afetam todas as subclasses, e hierarquias profundas ficam frágeis. Composição tende a produzir código mais flexível e testável.

**Regra prática:**

- Se a relação é realmente **"é um"** e a classe filha pode ser usada **em qualquer lugar** onde o pai é esperado → herança.
- Se você só quer **reaproveitar** algum comportamento, sem uma relação conceitual clara → **composição**.
- Na dúvida → **composição**.

### Exercício 4.1
Crie uma classe `Cupom` que representa um cupom de desconto, com atributos `codigo` (str) e `percentual` (float, ex: `10` para 10%). Adicione um método `aplicar(valor)` que retorna o valor com o desconto.

Em seguida, modifique `Pedido` para que ele **tenha um** `Cupom` opcional (composição). No método `total()`, se houver cupom, aplique o desconto antes de retornar. Teste com e sem cupom.

In [ ]:
# Escreva sua solução aqui



---
## ☕ Intervalo (15 min)

Até aqui vimos: herança simples, `super()`, sobrescrita, polimorfismo, *duck typing* e composição. Depois do intervalo: herança múltipla, MRO e classes abstratas — e a prática guiada.

---

## 5. Herança múltipla e MRO (*Method Resolution Order*)

Python, ao contrário de Java e C#, permite que uma classe herde de **mais de uma** classe pai simultaneamente. Isso se chama **herança múltipla**.

```python
class C(A, B):   # C herda de A E de B
    ...
```

Herança múltipla é uma ferramenta **poderosa**, mas também **perigosa**. Quando mal usada, gera o famoso "problema do diamante": se `A` e `B` definem um método com o mesmo nome, qual deles o `C` herda?

### 5.1 O exemplo mínimo

In [ ]:
class Grelhavel:
    """Componente: coisas que vão à grelha."""
    def preparar(self):
        print(f"  🔥 Jogando na grelha...")


class Embalavel:
    """Componente: coisas que são embaladas para viagem."""
    def preparar(self):
        print(f"  📦 Embalando para viagem...")


# XTudoToGo herda de AMBOS: é grelhável E embalável
class XTudoToGo(Grelhavel, Embalavel):
    pass


x = XTudoToGo()
x.preparar()     # Qual dos dois métodos é chamado?
print()
print(f"MRO: {[c.__name__ for c in XTudoToGo.__mro__]}")


O Python resolve o conflito usando o **MRO** (*Method Resolution Order*): ele procura o método na ordem em que as classes aparecem na declaração, da esquerda para a direita, subindo depois para as superclasses. O atributo `__mro__` mostra essa ordem.

Neste caso, como `Grelhavel` aparece antes de `Embalavel`, é o método de `Grelhavel` que é chamado. Se invertermos a ordem — `class XTudoToGo(Embalavel, Grelhavel)` — o método de `Embalavel` vence.

### 5.2 Quando usar herança múltipla?

Raramente. Na prática moderna, herança múltipla é usada principalmente através de **mixins** — classes pequenas e especializadas que adicionam um comportamento específico. Para a maioria dos casos, **composição é uma alternativa mais segura**.

> ⚠️ **Conselho:** se você está começando em POO, evite herança múltipla até ter muita experiência. É fácil criar hierarquias ambíguas e difíceis de depurar. Se sentir a tentação de usá-la, pergunte-se primeiro se composição não resolveria.

### Exercício 5.1 (rápido)
Experimente inverter a ordem de `XTudoToGo(Grelhavel, Embalavel)` para `XTudoToGo(Embalavel, Grelhavel)` e observe como o comportamento muda. Imprima o `__mro__` em cada caso.

In [ ]:
# Escreva sua solução aqui



---
## 6. Classes abstratas: estabelecendo contratos

Voltemos ao exemplo do polimorfismo. Tínhamos uma classe `Lanche` com um método `preparar()` "genérico" que era sobrescrito em cada subclasse. Mas olhe de perto:

In [ ]:
class Lanche:
    def __init__(self, nome, preco):
        self.nome = nome
        self.preco = preco

    def preparar(self):
        # Método "genérico" — mas o que significa "preparar de forma genérica"?
        print(f"  [{self.nome}] Preparando de forma genérica...")


# Nada impede que alguém instancie Lanche diretamente
generico = Lanche("Algo", 10.00)
generico.preparar()    # Funciona... mas não deveria!


O problema é conceitual: **não existe** "um lanche genérico" sendo preparado. Todo lanche é de algum tipo específico. A classe `Lanche` deveria ser apenas um **molde abstrato** — impossível de instanciar diretamente, e obrigando as subclasses a fornecerem sua própria implementação de `preparar()`.

Python oferece isso através do módulo `abc` (*abstract base classes*).

In [ ]:
from abc import ABC, abstractmethod


class Lanche(ABC):
    """Classe abstrata: não pode ser instanciada diretamente."""

    def __init__(self, nome, preco):
        self.nome = nome
        self.preco = preco

    def descrever(self):
        # Método NORMAL (não abstrato) — será herdado
        print(f"  🍔 {self.nome} — R${self.preco:.2f}")

    @abstractmethod
    def preparar(self):
        """Método abstrato: subclasses DEVEM implementar."""
        pass


class XTudo(Lanche):
    def __init__(self):
        super().__init__("X-Tudo", 18.50)

    def preparar(self):                        # ← obrigatório implementar
        print(f"  🔥 [CHAPA] {self.nome}: jogando na chapa quente!")


class LancheIncompleto(Lanche):
    """Esquece de implementar preparar() — vai falhar ao instanciar."""
    def __init__(self):
        super().__init__("Incompleto", 0)
    # (sem preparar())


# ─── Testes ─────────────────────────────────────────────────────────────
# 1. Tentar instanciar a classe abstrata diretamente → erro
try:
    generico = Lanche("Algo", 10.00)
except TypeError as e:
    print(f"✘ Classe abstrata: {e}")

# 2. Tentar instanciar subclasse que não implementou o método abstrato → erro
try:
    inc = LancheIncompleto()
except TypeError as e:
    print(f"✘ Subclasse incompleta: {e}")

# 3. Subclasse completa: funciona
x = XTudo()
x.descrever()
x.preparar()


**Vantagens das classes abstratas:**

1. **Contrato explícito**: você declara formalmente que toda subclasse *deve* implementar certos métodos.
2. **Erro precoce**: o erro aparece na instanciação, não na hora que o método é chamado — muito mais fácil de depurar.
3. **Documentação viva**: lendo a classe abstrata, fica claro o que é obrigação do programador da subclasse.

**Quando usar:**

- Quando há um conjunto de subclasses que precisam oferecer uma **interface comum**.
- Quando você quer **impedir** a instanciação da classe base "vazia".
- Quando o *duck typing* é inseguro demais para o contexto (ex.: código crítico com muitas extensões).

### ⚠️ Armadilha: métodos abstratos não precisam ter corpo vazio

O `pass` dentro do método abstrato é convenção, mas você pode colocar uma implementação "padrão" que as subclasses chamam via `super()`. Só não dá para *pular* a obrigação de sobrescrever.

### Exercício 6.1
Transforme `Bebida` (do Exercício 2.1) em uma classe abstrata. Marque o método `preparar()` como `@abstractmethod`. Confirme que tentar instanciar `Bebida` diretamente gera erro, e que `Refrigerante` / `Suco` funcionam normalmente.

In [ ]:
# Escreva sua solução aqui



---
## 7. Prática guiada: hierarquia de Funcionários do Podrão

Vamos juntar tudo — herança, sobrescrita, polimorfismo, classe abstrata — em um exemplo clássico: cálculo de **folha de pagamento**.

O Podrão tem três tipos de funcionário, cada um com uma regra diferente de salário:

- **Atendente**: salário fixo.
- **Cozinheiro**: salário fixo + adicional por insalubridade (10% do fixo).
- **Entregador**: salário base + R$2,50 por entrega feita no mês.

Todos os funcionários têm nome, CPF e mês de admissão, e todos devem responder ao método `calcular_salario()` — mas a *forma* de calcular depende do tipo. Esse é o cenário perfeito para uma **classe abstrata** com **polimorfismo**.

In [ ]:
from abc import ABC, abstractmethod


class Funcionario(ABC):
    """Classe abstrata: todo funcionário do Podrão tem nome e CPF,
    mas cada tipo calcula o salário de um jeito diferente."""

    def __init__(self, nome, cpf, salario_base):
        self.nome = nome
        self.cpf = cpf
        self.salario_base = salario_base

    @abstractmethod
    def calcular_salario(self):
        """Cada subclasse implementa sua própria regra."""
        pass

    def __str__(self):
        # Note: usa calcular_salario() da subclasse, mesmo sendo chamado
        # do método do pai — isso é polimorfismo dinâmico em ação.
        return f"{self.nome} ({type(self).__name__}) — R${self.calcular_salario():.2f}"


class Atendente(Funcionario):
    """Salário fixo, sem adicional."""
    def calcular_salario(self):
        return self.salario_base


class Cozinheiro(Funcionario):
    """Salário fixo + adicional de insalubridade de 10%."""
    INSALUBRIDADE = 0.10            # atributo de classe (constante)

    def calcular_salario(self):
        return self.salario_base * (1 + Cozinheiro.INSALUBRIDADE)


class Entregador(Funcionario):
    """Salário base + R$2,50 por entrega no mês."""
    VALOR_POR_ENTREGA = 2.50

    def __init__(self, nome, cpf, salario_base, entregas_mes=0):
        super().__init__(nome, cpf, salario_base)
        self.entregas_mes = entregas_mes

    def calcular_salario(self):
        return self.salario_base + self.entregas_mes * Entregador.VALOR_POR_ENTREGA


# ─── Folha de pagamento por polimorfismo ────────────────────────────────
equipe = [
    Atendente("Maria", "111.111.111-11", 1800.00),
    Cozinheiro("José", "222.222.222-22", 2200.00),
    Entregador("Pedro", "333.333.333-33", 1500.00, entregas_mes=120),
    Atendente("Ana", "444.444.444-44", 1800.00),
    Entregador("Bruno", "555.555.555-55", 1500.00, entregas_mes=95),
]

print("=== Folha de pagamento do Podrão ===")
total = 0.0
for func in equipe:
    print(f"  {func}")              # usa __str__ → usa calcular_salario() polimorficamente
    total += func.calcular_salario()
print(f"  {'─'*45}")
print(f"  TOTAL DA FOLHA: R${total:.2f}")

# Tentar instanciar Funcionario direto → erro
try:
    f = Funcionario("Teste", "000", 1000)
except TypeError as e:
    print(f"\n✘ Tentativa bloqueada: {e}")


Observe o que este exemplo demonstra de uma só vez:

1. **Classe abstrata** (`Funcionario(ABC)`) impede instanciação direta.
2. **Método abstrato** (`calcular_salario`) força cada subclasse a implementar sua regra.
3. **Herança** permite reaproveitar `__init__` e `__str__`.
4. **`super().__init__()`** em `Entregador` para lidar com atributo adicional.
5. **Polimorfismo** no laço da folha: o mesmo `calcular_salario()` dispara três regras diferentes.
6. **Atributos de classe** (`INSALUBRIDADE`, `VALOR_POR_ENTREGA`) para constantes compartilhadas.
7. **Polimorfismo dentro do `__str__` do pai**: mesmo escrito na classe abstrata, `self.calcular_salario()` chama a versão da subclasse. Isso se chama *late binding*.

Esse padrão (classe abstrata + método abstrato + polimorfismo) é a **espinha dorsal** de muitos sistemas orientados a objetos bem projetados.

---

## 8. Exercícios de consolidação

### Exercício 8.1 — Hierarquia de Formas
Modele uma hierarquia de formas geométricas:

- `Forma` (abstrata) com método abstrato `area()` e método concreto `descrever()` que imprime `"<NomeDaClasse> de área <valor>"`.
- `Circulo(Forma)` com atributo `raio` e implementação de `area()` (fórmula `π * r²`).
- `Retangulo(Forma)` com atributos `largura` e `altura`.
- `Quadrado(Retangulo)` — aproveite que um quadrado **é um** retângulo com largura = altura. Use `super().__init__(lado, lado)`.

Crie uma lista com várias formas e chame `descrever()` em cada uma via laço `for` (polimorfismo). Calcule também a soma das áreas.

In [ ]:
# Escreva sua solução aqui



### Exercício 8.2 — Composição revisitada
Crie uma classe `Motor` com atributos `potencia` (em cavalos) e método `ligar()` que imprime `"Motor de X cv ligado."`.

Em seguida, crie uma classe `Veiculo` que **tem um** motor (composição). O construtor de `Veiculo` deve receber o objeto motor e um nome. Adicione um método `ligar()` no veículo que delega para o motor (`self.motor.ligar()`).

Finalmente, crie `Carro(Veiculo)` por **herança**, acrescentando um atributo `num_portas`.

**Por que um misto?** Porque um `Carro` **é um** `Veiculo` (herança), mas um `Veiculo` **tem um** `Motor` (composição). Essa distinção é o coração do design orientado a objetos.

In [ ]:
# Escreva sua solução aqui



### Exercício 8.3 — Integrador (para casa): sistema de pagamentos
Modele um sistema de formas de pagamento usando classe abstrata + polimorfismo:

- `FormaPagamento` (abstrata) com atributo `taxa_transacao` (float) e método abstrato `processar(valor)` que deve retornar o valor líquido (após taxa).
- `Pix(FormaPagamento)` — taxa zero.
- `Cartao(FormaPagamento)` — taxa de 3% + atributo adicional `bandeira` (str).
- `Boleto(FormaPagamento)` — taxa fixa de R$2,50 + atributo `dias_vencimento` (int).

Adicione em cada uma:

- `__str__` retornando uma descrição amigável.
- `__eq__` comparando pela classe + atributos principais.

Crie uma função `processar_todos(pagamentos, valor)` que recebe uma lista de `FormaPagamento` e um valor, e imprime quanto cada uma líquida. Use essa função para comparar as três.

Entregue no Moodle até a próxima aula.

In [ ]:
# Escreva sua solução aqui (tarefa para casa)



---
## 9. Resumo da aula

| Conceito | Definição | Exemplo |
|---|---|---|
| **Herança** | Subclasse recebe atributos e métodos do pai | `class XTudo(Lanche):` |
| **`super()`** | Acessa a versão do pai de um método | `super().__init__(...)` |
| **Sobrescrita** *(override)* | Redefinir um método herdado | `def descrever(self): ...` |
| **Estensão** *(via super)* | Sobrescrever *chamando* o original | `super().descrever()` + extra |
| **Polimorfismo** | Mesmo método, comportamentos diferentes | `for item in itens: item.preparar()` |
| **Duck typing** | Python aceita qualquer objeto que responda ao método | Misturar `Lanche` e `Bebida` |
| **Composição** | Classe **tem um** outro objeto como atributo | `Pedido` tem `FormaPagamento` |
| **Herança múltipla** | Herdar de mais de uma classe | `class C(A, B):` |
| **MRO** | Ordem em que métodos são procurados | `C.__mro__` |
| **Classe abstrata** | Não pode ser instanciada; define contrato | `class Lanche(ABC):` |
| **`@abstractmethod`** | Força subclasses a implementarem | `@abstractmethod def preparar(self):` |

### Princípios a lembrar

1. **"É um" → herança; "tem um" → composição.** Na dúvida, composição.
2. **Prefira composição à herança** (Gang of Four). Hierarquias profundas são frágeis.
3. **Sempre chame `super().__init__()`** no início do `__init__` da subclasse.
4. Use `super().metodo()` dentro de sobrescritas para **estender em vez de substituir**.
5. Classes abstratas tornam contratos **explícitos** e falham **cedo** — úteis quando a segurança de tipos importa.
6. Herança múltipla existe, mas **use com moderação**. Mixins quando precisar.

### O que vem na Aula 4

Na última aula do módulo vamos cobrir **tópicos avançados e boas práticas**:

- **`@dataclass`** — reduzir *boilerplate* drasticamente.
- **`@classmethod` e `@staticmethod`** — construtores alternativos e utilitários.
- **Princípios SOLID** (introdução): SRP, OCP, LSP.
- **Padrões de projeto** clássicos em versão Pythônica (Strategy, Factory Method).
- **Estudo de caso integrador**: refatorar um sistema pequeno aplicando tudo que aprendemos.

Até lá!